In [1]:
!pip install --upgrade gdown
import gdown
file_id = '1djmNtqX6mVC1sDeaxmKxvFs0hTVjuxGI'
url = f'https://drive.google.com/uc?id={file_id}'
output = 'bhw2-data.zip'
gdown.download(url, output, quiet=False)

!unzip -q bhw2-data.zip -d ./data

Downloading...
From: https://drive.google.com/uc?id=1djmNtqX6mVC1sDeaxmKxvFs0hTVjuxGI
To: /kaggle/working/bhw2-data.zip
100%|██████████| 13.7M/13.7M [00:00<00:00, 47.1MB/s]


In [2]:
!pip install sacrebleu==2.3.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.0/57.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 5.8 MB/s eta 0:00:00


In [3]:
import math
import torch
import torch.nn as nn
import sacrebleu
import wandb
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

WANDB_API_KEY = ''
wandb.login(key=WANDB_API_KEY)

PAD_IDX = 0
BOS_TOKEN = '<bos>'
EOS_TOKEN = '<eos>'
UNK_TOKEN = '<unk>'
PAD_STR = '<pad>'

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: nnazarov (nnazarov-new-economic-school) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [4]:
class Vocabulary:
    def __init__(self, words_counter, min_freq=3):
        self.idx_to_str = [PAD_STR, BOS_TOKEN, EOS_TOKEN, UNK_TOKEN]
        self.str_to_idx = {word: i for i, word in enumerate(self.idx_to_str)}

        for word, count in words_counter.items():
            if count >= min_freq:
                self.str_to_idx[word] = len(self.idx_to_str)
                self.idx_to_str.append(word)

    def __len__(self):
        return len(self.idx_to_str)

    def text_to_indices(self, tokens):
        return [self.str_to_idx.get(w, self.str_to_idx[UNK_TOKEN]) for w in tokens]

    def indices_to_text(self, indices):
        return [self.idx_to_str[i] for i in indices]

def construct_vocab(filepath, min_freq=3):
    words_counter = Counter()
    with open(filepath, 'r', encoding='utf-8') as file:
        for text_line in file:
            words = text_line.strip().split()
            words_counter.update(words)
    return Vocabulary(words_counter, min_freq=min_freq)

class TranslationDataset(Dataset):
    def __init__(self, path_src, path_tgt, vocab_source, vocab_target):
        with open(path_src, 'r', encoding='utf-8') as fs:
            self.source_sentences = fs.readlines()
        with open(path_tgt, 'r', encoding='utf-8') as ft:
            self.target_sentences = ft.readlines()

        self.vocab_source = vocab_source
        self.vocab_target = vocab_target

    def __len__(self):
        return len(self.source_sentences)

    def __getitem__(self, index):
        src_words = self.source_sentences[index].strip().split()
        tgt_words = self.target_sentences[index].strip().split()

        encoded_src = self.vocab_source.text_to_indices(src_words)
        encoded_tgt = [self.vocab_target.str_to_idx[BOS_TOKEN]] + \
                      self.vocab_target.text_to_indices(tgt_words) + \
                      [self.vocab_target.str_to_idx[EOS_TOKEN]]

        return torch.tensor(encoded_src), torch.tensor(encoded_tgt)

def collate_fn(batch):
    sources, targets = zip(*batch)
    padded_sources = pad_sequence(sources, padding_value=PAD_IDX, batch_first=True)
    padded_targets = pad_sequence(targets, padding_value=PAD_IDX, batch_first=True)
    return padded_sources, padded_targets

In [5]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, d_ff, num_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_IDX)
        self.pos_encoding = PositionalEncoding(d_model, dropout=dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True
        )
        self.layers = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.d_model = d_model

    def forward(self, src, src_mask, src_padding_mask):
        x = self.embedding(src) * math.sqrt(self.d_model)
        x = self.pos_encoding(x)
        x = self.layers(x, mask=src_mask, src_key_padding_mask=src_padding_mask)
        return x

class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, d_ff, num_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_IDX)
        self.pos_encoding = PositionalEncoding(d_model, dropout=dropout)

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True
        )
        self.layers = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.fc_out = nn.Linear(d_model, vocab_size)
        self.d_model = d_model

    def forward(self, trg, encoder_output, trg_mask, trg_padding_mask, memory_padding_mask):
        x = self.embedding(trg) * math.sqrt(self.d_model)
        x = self.pos_encoding(x)
        x = self.layers(
            x, encoder_output,
            tgt_mask=trg_mask,
            tgt_key_padding_mask=trg_padding_mask,
            memory_key_padding_mask=memory_padding_mask
        )
        return self.fc_out(x)

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, trg, src_mask, trg_mask, src_padding_mask, trg_padding_mask):
        encoder_output = self.encoder(src, src_mask, src_padding_mask)
        output = self.decoder(
            trg, encoder_output,
            trg_mask, trg_padding_mask, src_padding_mask
        )
        return output

In [6]:
def make_attention_masks(src_tensor, tgt_tensor, device):
    src_pad_mask = (src_tensor == PAD_IDX)
    tgt_pad_mask = (tgt_tensor == PAD_IDX)
    seq_length = tgt_tensor.size(1)
    tgt_causal_mask = torch.triu(torch.ones(seq_length, seq_length), diagonal=1).bool().to(device)
    return src_pad_mask, tgt_pad_mask, tgt_causal_mask

def evaluate_model(model, val_dataloader, loss_fn, device):
    model.eval()
    accumulated_loss = 0
    with torch.no_grad():
        for src, tgt in val_dataloader:
            src, tgt = src.to(device), tgt.to(device)
            tgt_in = tgt[:, :-1]
            tgt_expected = tgt[:, 1:]
            
            s_pad_mask, t_pad_mask, c_mask = make_attention_masks(src, tgt_in, device)
            src_mask = None
            
            
            preds = model(src, tgt_in, src_mask, c_mask, s_pad_mask, t_pad_mask)
            loss = loss_fn(preds.reshape(-1, preds.size(-1)), tgt_expected.reshape(-1))
            accumulated_loss += loss.item()
    return accumulated_loss / len(val_dataloader)

def generate_translation(model, raw_sentence, v_src, v_tgt, device, max_tokens=80):
    model.eval()
    tokens = raw_sentence.strip().split()
    src_tensor = torch.tensor([v_src.text_to_indices(tokens)]).to(device)
    s_mask = (src_tensor == PAD_IDX)
    src_mask = None
    
    with torch.no_grad():
        
        enc_out = model.encoder(src_tensor, src_mask, s_mask)
        
    generated_indices = [v_tgt.str_to_idx[BOS_TOKEN]]
    for _ in range(max_tokens):
        tgt_tensor = torch.tensor([generated_indices]).to(device)
        c_mask = torch.triu(torch.ones(len(generated_indices), len(generated_indices)), diagonal=1).bool().to(device)
        
        with torch.no_grad():
            
            dec_out = model.decoder(tgt_tensor, enc_out, c_mask, None, s_mask)
            
        pred_token = dec_out[0, -1].argmax().item()
        generated_indices.append(pred_token)
        if pred_token == v_tgt.str_to_idx[EOS_TOKEN]:
            break
    return ' '.join(v_tgt.indices_to_text(generated_indices[1:-1]))

def score_bleu(model, src_sentences, ref_sentences, v_src, v_tgt, device):
    model.eval()
    hypotheses = []
    for s_sent in src_sentences:
        hypotheses.append(generate_translation(model, s_sent, v_src, v_tgt, device))
    metric = sacrebleu.corpus_bleu(hypotheses, [ref_sentences], tokenize='none', force=True)
    return metric.score

In [7]:
PATH_SRC_TRAIN = '/kaggle/working/data/data/train.de-en.de'
PATH_TGT_TRAIN = '/kaggle/working/data/data/train.de-en.en'
PATH_SRC_VAL = '/kaggle/working/data/data/val.de-en.de'
PATH_TGT_VAL = '/kaggle/working/data/data/val.de-en.en'

vocab_source = construct_vocab(PATH_SRC_TRAIN, min_freq=3)
vocab_target = construct_vocab(PATH_TGT_TRAIN, min_freq=3)

CONFIG = {
    'd_model': 512,
    'n_heads': 8,
    'd_ff': 512,
    'n_layers': 3,
    'dropout': 0.1,
    'batch_size': 128,
    'epochs': 25,
    'learning_rate': 1e-4,
    'vocab_src_size': len(vocab_source),
    'vocab_tgt_size': len(vocab_target)
}

wandb.init(project='DL_BHW222', config=CONFIG)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

encoder = Encoder(CONFIG['vocab_src_size'], CONFIG['d_model'], CONFIG['n_heads'], CONFIG['d_ff'], CONFIG['n_layers'], CONFIG['dropout'])
decoder = Decoder(CONFIG['vocab_tgt_size'], CONFIG['d_model'], CONFIG['n_heads'], CONFIG['d_ff'], CONFIG['n_layers'], CONFIG['dropout'])
decoder.fc_out.weight = decoder.embedding.weight

translation_model = Seq2Seq(encoder, decoder)
for param in translation_model.parameters():
    if param.dim() > 1:
        nn.init.xavier_uniform_(param)
translation_model = translation_model.to(device)

ds_train = TranslationDataset(PATH_SRC_TRAIN, PATH_TGT_TRAIN, vocab_source, vocab_target)
ds_val = TranslationDataset(PATH_SRC_VAL, PATH_TGT_VAL, vocab_source, vocab_target)

loader_train = DataLoader(ds_train, batch_size=CONFIG['batch_size'], shuffle=True, collate_fn=collate_fn)
loader_val = DataLoader(ds_val, batch_size=CONFIG['batch_size'], shuffle=False, collate_fn=collate_fn)

with open(PATH_TGT_VAL, 'r', encoding='utf-8') as f:
    eval_references = [l.strip() for l in f]
with open(PATH_SRC_VAL, 'r', encoding='utf-8') as f:
    eval_sources = [l.strip() for l in f]

optimizer = torch.optim.Adam(translation_model.parameters(), lr=CONFIG['learning_rate'], betas=(0.9, 0.98), eps=1e-9)
loss_function = nn.CrossEntropyLoss(ignore_index=PAD_IDX, label_smoothing=0.1)
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['epochs'], eta_min=1e-6)
grad_scaler = torch.amp.GradScaler('cuda')

wandb: Tracking run with wandb version 0.24.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260316_184557-0lh65u78
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run resilient-sun-3
wandb: ⭐️ View project at https://wandb.ai/nnazarov-new-economic-school/DL_BHW222
wandb: 🚀 View run at https://wandb.ai/nnazarov-new-economic-school/DL_BHW222/runs/0lh65u78


In [8]:
from tqdm.auto import tqdm

top_bleu = 0.0

wandb.watch(translation_model, log='all', log_freq=100)
epochs_to_run = 15 #15 эпох, тк не успеваю
train_pbar = tqdm(range(epochs_to_run), desc='Training Progress')

for epoch_idx in train_pbar:
    translation_model.train()
    epoch_loss = 0.0

    batch_pbar = tqdm(loader_train, desc=f'Epoch {epoch_idx+1}', leave=False)

    for b_src, b_tgt in batch_pbar:
        b_src, b_tgt = b_src.to(device), b_tgt.to(device)

        tgt_input = b_tgt[:, :-1]
        tgt_labels = b_tgt[:, 1:]

        s_pad_mask, t_pad_mask, t_causal_mask = make_attention_masks(b_src, tgt_input, device)
        src_mask = None

        optimizer.zero_grad()
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            
            logits = translation_model(b_src, tgt_input, src_mask, t_causal_mask, s_pad_mask, t_pad_mask)
            loss_val = loss_function(logits.reshape(-1, logits.size(-1)), tgt_labels.reshape(-1))

        grad_scaler.scale(loss_val).backward()
        grad_scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(translation_model.parameters(), max_norm=1.0)
        grad_scaler.step(optimizer)
        grad_scaler.update()

        current_loss = loss_val.item()
        epoch_loss += current_loss

        batch_pbar.set_postfix({'batch_loss': f'{current_loss:.4f}'})

    avg_train_loss = epoch_loss / len(loader_train)
    avg_val_loss = evaluate_model(translation_model, loader_val, loss_function, device)
    current_bleu = score_bleu(translation_model, eval_sources, eval_references, vocab_source, vocab_target, device)

    lr_scheduler.step()

    wandb.log({
        'epoch': epoch_idx + 1,
        'train_loss': avg_train_loss,
        'val_loss': avg_val_loss,
        'bleu_score': current_bleu,
        'learning_rate': optimizer.param_groups[0]['lr']
    })

    if current_bleu > top_bleu:
        top_bleu = current_bleu
        torch.save(translation_model.state_dict(), 'best_translation_model.pt')

    train_pbar.set_postfix({'best_bleu': f'{top_bleu:.2f}', 'val_loss': f'{avg_val_loss:.4f}'})

    print(f'\nEpoch {epoch_idx+1}/{CONFIG['epochs']}')
    print(f'Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | BLEU: {current_bleu:.2f} (Best: {top_bleu:.2f})')

    sample_out = generate_translation(translation_model, eval_sources[0], vocab_source, vocab_target, device)
    print(f'Target: {eval_references[0]}')
    print(f'Pred  : {sample_out}\n')

    torch.save(translation_model.state_dict(), f'model_ckpt_ep{epoch_idx+1}.pt')
    torch.cuda.empty_cache()

wandb.finish()
print(f'Обучение завершено. Лучший BLEU: {top_bleu:.2f}')

Training Progress:   0%|          | 0/15 [00:00<?, ?it/s]

Epoch 1:   0%|          | 0/1531 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:515: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(



Epoch 1/25
Train Loss: 5.6458 | Val Loss: 4.6750 | BLEU: 7.87 (Best: 7.87)
Target: when i was 11 , i remember waking up one morning to the sound of joy in my house .
Pred  : when i was my father , i was a year of the united states of the united states .



Epoch 2:   0%|          | 0/1531 [00:00<?, ?it/s]

...


Epoch 14/25
Train Loss: 3.0977 | Val Loss: 3.2535 | BLEU: 31.29 (Best: 31.46)
Target: when i was 11 , i remember waking up one morning to the sound of joy in my house .
Pred  : when i was 11 years old , i was sort of drawn into the sound of the sound .



Epoch 15:   0%|          | 0/1531 [00:00<?, ?it/s]


Epoch 15/25
Train Loss: 3.0693 | Val Loss: 3.2439 | BLEU: 31.67 (Best: 31.67)
Target: when i was 11 , i remember waking up one morning to the sound of joy in my house .
Pred  : when i was 11 years old , i was kind of drawn into the sound of the sound .



wandb: updating run metadata
wandb: uploading history steps 14-14, summary, console lines 85-90
wandb: 
wandb: Run history:
wandb:    bleu_score ▁▃▅▆▆▇▇▇███████
wandb:         epoch ▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
wandb: learning_rate ███▇▇▇▆▆▅▄▄▃▂▂▁
wandb:    train_loss █▅▄▃▃▂▂▂▂▂▁▁▁▁▁
wandb:      val_loss █▅▄▃▂▂▂▁▁▁▁▁▁▁▁
wandb: 
wandb: Run summary:
wandb:    bleu_score 31.67003
wandb:         epoch 15
wandb: learning_rate 4e-05
wandb:    train_loss 3.06929
wandb:      val_loss 3.24394
wandb: 
wandb: 🚀 View run resilient-sun-3 at: https://wandb.ai/nnazarov-new-economic-school/DL_BHW222/runs/0lh65u78
wandb: ⭐️ View project at: https://wandb.ai/nnazarov-new-economic-school/DL_BHW222
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260316_184557-0lh65u78/logs


Обучение завершено. Лучший BLEU: 31.67


In [9]:
import os
from tqdm.auto import tqdm

def generate_translation(model, raw_sentence, v_src, v_tgt, device, max_tokens=80):
    model.eval()
    
    tokens = raw_sentence.strip().split()
    if not tokens:
        return ''
    
    src_indices = v_src.text_to_indices(tokens)
    src_tensor = torch.tensor([src_indices]).to(device)

    src_padding_mask = (src_tensor == PAD_IDX)
    src_mask = None 
    
    with torch.no_grad():
        memory = model.encoder(src_tensor, src_mask, src_padding_mask)
        
    generated_indices = [v_tgt.str_to_idx[BOS_TOKEN]]
    
    for _ in range(max_tokens):
        tgt_tensor = torch.tensor([generated_indices]).to(device)
        
        sz = len(generated_indices)
        tgt_mask = torch.triu(torch.ones(sz, sz), diagonal=1).bool().to(device)
        
        with torch.no_grad():
            output = model.decoder(
                tgt_tensor, 
                memory, 
                tgt_mask, 
                None,
                src_padding_mask
            )
            
        next_token = output[0, -1].argmax().item()
        generated_indices.append(next_token)
        
        if next_token == v_tgt.str_to_idx[EOS_TOKEN]:
            break
            
    return ' '.join(v_tgt.indices_to_text(generated_indices[1:-1]))

def save_predictions(model, src_filepath, output_filepath, v_src, v_tgt, device):
    model.eval()
    
    if not os.path.exists(src_filepath):
        print(f'Ошибка: Файл {src_filepath} не найден!')
        return

    with open(src_filepath, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    print(f'Начинаю перевод {len(lines)} предложений')

    with open(output_filepath, 'w', encoding='utf-8') as out_f:
        for line in tqdm(lines, desc="Translating"):
            translated_text = generate_translation(model, line, v_src, v_tgt, device)
            out_f.write(translated_text + '\n')

    print(f'Все топ, готово! Результат сохранен в: {output_filepath}')

if os.path.exists('best_translation_model.pt'):
    translation_model.load_state_dict(torch.load('best_translation_model.pt', map_location=device))
    print('Загружены лучшие веса модели')

PATH_TEST_SRC = '/kaggle/working/data/data/test1.de-en.de'

save_predictions(
    translation_model,
    PATH_TEST_SRC,
    'answers.txt',
    vocab_source,
    vocab_target,
    device
)

Загружены лучшие веса модели
Начинаю перевод 2998 предложений


Translating:   0%|          | 0/2998 [00:00<?, ?it/s]

Все топ, готово! Результат сохранен в: answers.txt
